# Revision (Person B, L4): DAPT replicates for RoBERTa / T5-small / T5-base, then MLM-scoring
Run Cell 1 + Cell 2 from `B_week3.ipynb` first (restart between). Cell 2 must show **L4, transformers 4.46.3, numpy 2.1.3**.
Every loop skips runs whose JSON exists, so rerun the same cell after a disconnect.

CPU tasks B1-B3 are already done locally (logs in `logs/check_labels_revision.log`, `logs/per_class.log`, `logs/n_params.log`).
Do not start B4 until A's analysis rule is approved.

## B0: pre-flight. Replicates run at `01f54eb`, the same commit as the GPT-2 replicates, so `git_commit` matches.

In [ ]:
!echo "SMOKE=[$SMOKE] DAPT_BS=[$DAPT_BS]"        # both must be empty
!git fetch -q && git checkout -q 01f54eb && git log --oneline -1   # expect 01f54eb
!git status --short                                  # must be clean
SHARED = '/content/drive/MyDrive/dapt-absa-shared'
!rm -rf runs_extra && ln -s {SHARED}/runs_extra runs_extra
!ls -la runs_extra | head -1; ls runs_extra | wc -l  # symlink to Drive; 40+ files already there
!ls checkpoints/
!df -h /content/drive | tail -1                      # need ~3.5 GB free on Drive
!mkdir -p logs

## B4: DAPT replicates. Set ARCH, run 4.1 -> 4.2 (post, wait for C's PASS) -> 4.3. Order: `roberta`, `t5small`, `t5`.
Expected `loss_before`: roberta 1.897, t5small 2.796, t5 2.273. Watch T5 for `nan`. After a disconnect, look for `RESUMING from step N`.

In [ ]:
ARCH = "roberta"          # then "t5small", then "t5" 

In [ ]:
# 4.1 DAPT d1, d2
for D in [1, 2]:
    !python -u -m src.dapt --arch {ARCH} --seed {D} 2>&1 | tee -a logs/dapt_{ARCH}_d{D}.log

In [ ]:
# 4.2 Gate 2 snippet (B5): post this printout for C's PASS
import json, glob, math
for f in sorted(glob.glob("checkpoints/*_dapt_d*/dapt_report.json")):
    r = json.load(open(f))
    before, after = r["loss_before"], r["loss_after"]
    drop = before - after
    ok = math.isfinite(after) and drop > 0 and r["tokens"] >= 0.99 * 50_000_000
    print(f.split("/")[1], f"seed={r.get('seed')} before={before:.3f} after={after:.3f} "
          f"drop={drop:+.3f} tokens={r['tokens']:,}", "PASS" if ok else "FAIL")

In [ ]:
# 4.3 fine-tunes: 2 checkpoints x 5 seeds (only after C's PASS)
import os
for D in [1, 2]:
    for S in [13, 42, 1337, 7, 2024]:
        out = f"runs_extra/{ARCH}_dapt_d{D}_s{S}.json"
        if os.path.exists(out):
            print("skip", out); continue
        !python -u -m src.train_absa --arch {ARCH} --init dapt --dapt-seed {D} --seed {S} 2>&1 | tee logs/rep_{ARCH}_d{D}_s{S}.log
!ls runs_extra/{ARCH}_dapt_d*_s*.json | wc -l      # want 10

## B6: extras.py after all three architectures. It already loops every arch; expect `dapt=d1` and `dapt=d2` lines at `lr x1`, `n=5`, for roberta, t5, t5small and gpt2.

In [ ]:
!git checkout -q main && git pull -q && git log --oneline -1   # expect e12c238 or later: extras.py with scoring-aware pairing
!PYTHONPATH=src python scripts/extras.py 2>&1 | tee logs/extras_revision.log

## B7: RoBERTa MLM-head label scoring (D12 Opt 2). Code is on `main`; real runs only after A and C approve it.
Smoke first: loss must fall and epoch-1 val F1 must be clearly above 33%. (Local CPU check already gave 37% zero-shot, 66% after 256 training rows.)

In [ ]:
%env SMOKE=1
!python -u -m src.train_absa --arch roberta --init base --seed 42 --scoring mlm 2>&1 | tee logs/smoke_mlmscore.log
%env SMOKE=

In [ ]:
# 20 real runs, ~3 h. --init dapt uses the original checkpoints/roberta_dapt (already passed Gate 2)
!echo "SMOKE=[$SMOKE]"; git log --oneline -1
import os
for init in ["base", "dapt"]:
    for S in [13, 42, 1337, 7, 2024]:
        out = f"runs_extra/roberta_{init}_s{S}_mlmscore.json"
        if os.path.exists(out):
            print("skip", out); continue
        !python -u -m src.train_absa --arch roberta --init {init} --seed {S} --scoring mlm 2>&1 | tee logs/mlmscore_{init}_s{S}.log

In [ ]:
!PYTHONPATH=src python scripts/extras.py 2>&1 | tee logs/extras_mlmscore.log | grep -A1 mlmscore   # post this to C